In [1]:
import qubx
from qubx import logger

%qubxd

%load_ext autoreload
%autoreload 2

from qubx.backtester.simulator import simulate
from qubx.core.basics import DataType, Signal
from qubx.core.interfaces import IStrategy, IStrategyContext, IStrategyInitializer, MarketEvent, TriggerEvent
from qubx.core.lookups import lookup
from qubx.data.registry import StorageRegistry
from qubx.data.transformers import TypedGenericSeries
from qubx.utils.time import handle_start_stop


⠀⠀⡰⡖⠒⠒⢒⢦⠀⠀   
⠀⢠⠃⠈⢆⣀⣎⣀⣱⡀  QUBX | Quantitative Backtesting Environment 
⠀⢳⠒⠒⡞⠚⡄⠀⡰⠁         (c) 2025, ver. 0.7.40.dev8
⠀⠀⠱⣜⣀⣀⣈⣦⠃⠀⠀⠀ 
        


## Datareaders 

In [3]:
stor = StorageRegistry.get("csv::../../tests/data/storages/csv/")
stor.get_exchanges()

['BINANCE.UM', 'HYPERLIQUID', 'KRAKEN']

In [ ]:
stor.get_reader("BINANCE.UM", "SWAP").get_time_range("AAVEUSDT", "ohlc(1h)")

In [ ]:
stor.get_reader("BINANCE.UM", "SWAP").get_time_range("BTCUSDT", "quote")

In [ ]:
stor.get_reader("KRAKEN", "SWAP").get_time_range("AAVEUSD", "ohlc(1h)")

In [ ]:
stor.get_reader("KRAKEN", "SWAP").read("BTCUSD", "quote").to_records()[:3]

In [ ]:
stor.get_reader("KRAKEN", "SWAP").read("BTCUSD", "quote", None, None).transform(TypedGenericSeries())

In [11]:
_X1 = pd.DataFrame.from_dict({
    "2025-01-01 00:00": {"open": 1000, "high": 1100, "low": 950, "close": 1050 },
    "2025-01-01 01:00": {"open": 1050, "high": 1200, "low": 900, "close": 1075 },
    "2025-01-01 02:00": {"open": 1075, "high": 1300, "low": 1050, "close": 1100 },
}, orient="index")
_X1.index = pd.DatetimeIndex(_X1.index)

_X2 = pd.DataFrame.from_dict({
    "2025-01-01 00:00": {"open": 100, "high": 110, "low": 95, "close": 105 },
    "2025-01-01 01:00": {"open": 105, "high": 120, "low": 90, "close": 107 },
    "2025-01-01 02:00": {"open": 107, "high": 130, "low": 105, "close": 110 },
}, orient="index")
_X2.index = pd.DatetimeIndex(_X2.index)

In [ ]:
stor = StorageRegistry.get("handy", data={
    "BINANCE.UM:SWAP:BTCUSDT": _X1,
    "BINANCE.UM:SWAP:ETHUSDT": _X2
})
r = stor["BINANCE.UM", "SWAP"]
r.read(["BTCUSDT", "ETHUSDT"], "ohlc(1h)")

## Smoke test run

In [52]:
stor = StorageRegistry.get("csv::../../tests/data/storages/multi/")
stor.get_exchanges()

['BINANCE.UM', 'KRAKEN.F']

In [53]:
class Test1(IStrategy):
    def on_init(self, initializer: IStrategyInitializer):
        # initializer.set_base_subscription("ohlc(1h)")
        # initializer.set_base_subscription("ohlc_quotes(1h)")
        # initializer.subscribe("ohlc_trades(1h)")
        # initializer.set_event_schedule("1h")
        initializer.set_base_subscription("quote")

    def on_market_data(self, ctx: IStrategyContext, data: MarketEvent) -> list[Signal] | Signal | None:
        # qq = ''
        # for i in ctx.instruments:
        #     qq += f" | {i}: {ctx.quote(i).mid_price()}"

        logger.info(data)

    # def on_event(self, ctx: IStrategyContext, event: TriggerEvent) -> list[Signal] | Signal | None:
    #     qq = ''
    #     for i in ctx.instruments:
    #         qq += f" | {i}: {ctx.quote(i).mid_price()}"

    #     logger.info(qq)

In [54]:
simulate(
    Test1(), 
    data=stor, capital=1000, 
    # start="2023-06-01", stop="2023-06-02",
    start="2025-07-01", stop="2025-07-02",
    instruments=[
        "BINANCE.UM:SWAP:BTCUSDT",
        # "KRAKEN.F:SWAP:BTCUSD"
    ], 
    debug="INFO"
)

2026-02-11 16:49:38.611 [ ℹ️ ] (data) SimulatedDataProvider.BINANCE.UM is initialized
2026-02-11 16:49:38.612 [ ⚠️ ] (runner) qubx.backtester.runner:_handle_ctx_subscriptions:512 - [simulator] :: Event schedule is not specified.
 - Only on_market_data() will be triggered !
 - To enable on_event() call initializer.set_event_schedule(...) in on_init()
2025-07-01 00:00:00.000 [ℹ️] (runner) SimulationRunner ::: Simulation started at 2025-07-01 00:00:00 :::


Simulating:   0%|          | 0/100 [00:00<?, ?%/s]

2025-07-01 00:00:00.000 [ℹ️] (context) [StrategyContext] :: Strategy context stopped
2025-07-01 00:00:00.000 [❌] (simulator) Simulation setup 0 failed with error: Can't find any csv data for 'BTCUSDT' of 'quote' in ../../tests/data/storages/multi/BINANCE.UM/SWAP !
2025-07-01 00:00:00.000 [❌] (simulator) Traceback (most recent call last):
  File "/home/quant0/devs/Qubx/src/qubx/backtester/simulator.py", line 281, in _run_setup
    runner.run(silent=silent, close_data_readers=close_data_readers)
  File "/home/quant0/devs/Qubx/src/qubx/backtester/runner.py", line 156, in run
    raise e
  File "/home/quant0/devs/Qubx/src/qubx/backtester/runner.py", line 150, in run
    self._run(self.start, stop, silent=silent)
  File "/home/quant0/devs/Qubx/src/qubx/backtester/runner.py", line 285, in _run
    for instrument, data_type, event, is_hist in qiter:
                                                 ^^^^^
  File "/home/quant0/devs/Qubx/src/qubx/backtester/simulated_data.py", line 1112, in __ite

[]

2025-07-01 00:00:00.000 [⚠️] (simulator) qubx.backtester.simulator:_run_setups:221 - Simulation setup 0 failed - skipping from results


In [19]:
r = StorageRegistry.get("qdb::quantlab")["BINANCE.UM", "SWAP"]

In [23]:
r.get_time_range("BTCUSDT", "quote")

(numpy.datetime64('2025-07-01T00:00:01.000000'),
 numpy.datetime64('2026-02-11T00:00:00.000000'))

In [27]:
qx = r.read("BTCUSDT", "quote", "2025-07-01", "2025-07-02")

In [51]:
qx.to_pd().drop(columns=["symbol"]).to_csv("../../tests/data/storages/multi/BINANCE.UM/SWAP/NTCUSDT.quote.csv.gz")

In [ ]:
qqs

                          bid       ask  bid_size  ask_size
time                                                       
2025-07-01 00:00:01  107087.3  107087.4    10.907     9.117
2025-07-01 00:00:02  107087.3  107087.4    10.909     9.117
2025-07-01 00:00:05  107087.3  107087.4     2.748     7.339
2025-07-01 00:00:06  107079.3  107079.4    15.008     2.689
2025-07-01 00:00:07  107079.3  107079.4    14.473     4.164
...                       ...       ...       ...       ...
2025-07-01 23:59:55  105637.9  105638.0    23.883     0.580
2025-07-01 23:59:56  105637.9  105638.0    23.500     0.584
2025-07-01 23:59:57  105637.9  105638.0    20.324     0.572
2025-07-01 23:59:58  105637.9  105638.0     5.035     9.885
2025-07-01 23:59:59  105637.9  105638.0     5.713     7.374

[86371 rows x 4 columns]